# Clean vs. unclean flavour events: y_23 and jet multiplicity

Overlays of event variables for events with a clean parton-level flavour signature (IFN) and events without,
from the `<variable>_{clean,unclean}` histograms written by `jet-flv-test-histmaker.py`, to look for a cut that
removes the unclean events.

* Clean: H → qq̄ jets with q content ±1 and no other flavour; H → gg jets with no flavour at all.
  The classification is always done at **parton** level; the variables are shown for the parton- and the
  hadron-level clustering inputs.
* Variables:
  * `y23`: log10(y_23), y_23 = d_23 / E_vis² of the exclusive 2-jet ee_kt clustering (identical for IFN and net
    summation). Events with y_23 = 0 (only 2 partons) are in the underflow bin.
  * `njets_emin0`, `njets_emin5`: number of inclusive anti-kt jets (ee_genkt, R = 0.5, p = -1) without an energy
    cut and with E > 5 GeV.
* Top pad: both curves normalised to the total number of events in the sample, so their relative size shows
  how rare the unclean events are. Bottom pad: for a cut "keep x < cut", the fraction of clean events kept, of
  unclean events rejected, and the clean fraction (purity) of the kept events. Bins have integer edges, an integer
  value n is in [n, n+1), so "keep N_jets < 3" means N_jets <= 2.
* Output: `plots/<level>_ifn/<variable>/<sample>.pdf` and `plots/<level>_ifn/<variable>/all_samples_<variable>.pdf`.

In [1]:
import os
import glob
import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)

INPUT_DIR = os.path.join("..", "..", "..", "outputs", "jet-test")
PLOT_DIR = os.path.join(INPUT_DIR, "plots")
LEVELS = ["parton", "hadron"]

# variable -> histogram name (without the _clean/_unclean suffix), x title, x range shown, cuts for the table
VARIABLES = {
    "y23": dict(hist="{level}_ifn_log10_y23", xtitle="log_{10}(y_{23})", xrange=(-6., 0.),
                cuts=[-3., -2.5, -2., -1.5, -1.]),
    "njets_emin0": dict(hist="{level}_incl_emin0_njets", xtitle="N_{jets} (ee_genkt R=0.5, no E cut)",
                        xrange=(0., 20.), cuts=[3., 4., 5., 6., 7.]),
    "njets_emin5": dict(hist="{level}_incl_emin5_njets", xtitle="N_{jets} (ee_genkt R=0.5, E > 5 GeV)",
                        xrange=(0., 10.), cuts=[2., 3., 4., 5.]),
}

files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.root")))
files

['../../../outputs/jet-test/wzp6_ee_nunuH_Hbb_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Hcc_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Hdd_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Hgg_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Hss_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Huu_ecm240.root']

In [2]:
def cut_scan(h_clean, h_unclean):
    """Clean efficiency, unclean rejection and clean purity for a cut x < upper bin edge (underflow included)."""
    n_c = h_clean.Integral(0, h_clean.GetNbinsX() + 1)
    n_u = h_unclean.Integral(0, h_unclean.GetNbinsX() + 1)
    nb = h_clean.GetNbinsX()
    eff = ROOT.TGraph()
    rej = ROOT.TGraph()
    pur = ROOT.TGraph()
    for i in range(0, nb + 1):
        x = h_clean.GetXaxis().GetBinUpEdge(i) if i > 0 else h_clean.GetXaxis().GetXmin()
        eff.SetPoint(eff.GetN(), x, h_clean.Integral(0, i) / n_c if n_c else 0.)
        rej.SetPoint(rej.GetN(), x, 1. - h_unclean.Integral(0, i) / n_u if n_u else 0.)
        kept = h_clean.Integral(0, i) + h_unclean.Integral(0, i)
        if kept > 0:
            pur.SetPoint(pur.GetN(), x, h_clean.Integral(0, i) / kept)
    return eff, rej, pur, n_c, n_u

In [3]:
def draw(tfile, sample, level, var, cfg):
    name = cfg["hist"].format(level=level)
    h_c = tfile.Get(f"{name}_clean").Clone(f"{sample}_{level}_{var}_c")
    h_u = tfile.Get(f"{name}_unclean").Clone(f"{sample}_{level}_{var}_u")
    for h in (h_c, h_u):
        h.SetDirectory(0)
    eff, rej, pur, n_c, n_u = cut_scan(h_c, h_u)
    n_tot = n_c + n_u
    uf_c = h_c.GetBinContent(0) / n_c if n_c else 0.
    uf_u = h_u.GetBinContent(0) / n_u if n_u else 0.
    xmin, xmax = cfg["xrange"]

    c = ROOT.TCanvas(f"{level}_{var}_{sample}", "", 700, 750)
    top = ROOT.TPad("top", "", 0, 0.38, 1, 1)
    bot = ROOT.TPad("bot", "", 0, 0, 1, 0.38)
    top.SetBottomMargin(0.02)
    bot.SetTopMargin(0.03)
    bot.SetBottomMargin(0.25)
    for p in (top, bot):
        p.SetGridx()
        p.Draw()

    top.cd()
    for h, col in ((h_c, ROOT.kBlue + 1), (h_u, ROOT.kRed + 1)):
        h.Scale(1. / n_tot)
        h.SetLineColor(col)
        h.SetLineWidth(2)
    h_c.SetTitle(f"{sample}, {level} level;;fraction of all events")
    h_c.GetXaxis().SetLabelSize(0)
    h_c.GetXaxis().SetRangeUser(xmin, xmax)
    h_c.SetMaximum(1.4 * max(h_c.GetMaximum(), h_u.GetMaximum()))
    h_c.Draw("hist")
    h_u.Draw("hist same")
    # underflow is only populated for y_23 = 0
    uf = lambda f: f", underflow: {f:.1%}" if var == "y23" else ""
    leg = ROOT.TLegend(0.4, 0.68, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    leg.SetTextSize(0.045)
    leg.AddEntry(h_c, f"clean parton flavour ({n_c / n_tot:.1%}{uf(uf_c)})", "l")
    leg.AddEntry(h_u, f"not clean ({n_u / n_tot:.1%}{uf(uf_u)})", "l")
    leg.Draw()

    bot.cd()
    frame = bot.DrawFrame(xmin, 0, xmax, 1.05)
    frame.SetTitle(f";{cfg['xtitle']} cut (keep < cut);fraction")
    for ax, s in ((frame.GetXaxis(), 0.08), (frame.GetYaxis(), 0.08)):
        ax.SetLabelSize(s)
        ax.SetTitleSize(s)
    frame.GetXaxis().SetTitleOffset(1.2)
    frame.GetYaxis().SetTitleOffset(0.5)
    frame.GetYaxis().SetNdivisions(505)
    eff.SetLineColor(ROOT.kBlue + 1)
    rej.SetLineColor(ROOT.kRed + 1)
    pur.SetLineColor(ROOT.kGreen + 2)
    pur.SetLineStyle(2)
    for g in (eff, rej, pur):
        g.SetLineWidth(2)
        g.Draw("L same")
    leg2 = ROOT.TLegend(0.55, 0.3, 0.88, 0.65)
    leg2.SetBorderSize(0)
    leg2.SetFillStyle(0)
    leg2.SetTextSize(0.08)
    leg2.AddEntry(eff, "clean kept", "l")
    leg2.AddEntry(rej, "not clean rejected", "l")
    leg2.AddEntry(pur, "clean fraction of kept", "l")
    leg2.Draw()

    outdir = os.path.join(PLOT_DIR, f"{level}_ifn", var)
    os.makedirs(outdir, exist_ok=True)
    c.SaveAs(os.path.join(outdir, f"{sample}.pdf"))
    c.Draw()
    c._keep = (top, bot, h_c, h_u, eff, rej, pur, leg, leg2, frame)
    return c, eff, rej, pur, n_c, n_u

In [4]:
canvases = {}
scans = {}
for fname in files:
    sample = os.path.basename(fname)[:-5]
    tfile = ROOT.TFile.Open(fname)
    for var, cfg in VARIABLES.items():
        for level in LEVELS:
            c, eff, rej, pur, n_c, n_u = draw(tfile, sample, level, var, cfg)
            canvases.setdefault((level, var), []).append(c)
            scans[(var, sample, level)] = (eff, rej, n_c, n_u)
    tfile.Close()

for (level, var), cs in canvases.items():
    combined = os.path.join(PLOT_DIR, f"{level}_ifn", var, f"all_samples_{var}.pdf")
    cs[0].Print(combined + "[")
    for c in cs:
        c.Print(combined, f"Title:{c.GetName()}")
    cs[-1].Print(combined + "]")

Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/y23/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created


Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/y23/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_emin0/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_emin0/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_emin5/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_emin5/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/y23/wzp6_ee_nunuH_Hcc_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/y23/wzp6_ee_nunuH_Hcc_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parto

Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/y23/wzp6_ee_nunuH_Hdd_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_emin0/wzp6_ee_nunuH_Hdd_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_emin0/wzp6_ee_nunuH_Hdd_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_emin5/wzp6_ee_nunuH_Hdd_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_emin5/wzp6_ee_nunuH_Hdd_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/y23/wzp6_ee_nunuH_Hgg_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/y23/wzp6_ee_nunuH_Hgg_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parto

Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_emin0/all_samples_njets_emin0.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_emin0/all_samples_njets_emin0.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_emin0/all_samples_njets_emin0.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_emin0/all_samples_njets_emin0.pdf
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_emin0/all_samples_njets_emin0.pdf has been closed
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_emin5/all_samples_njets_emin5.pdf has been created
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_emin5/all_samples_njets_emin5.pdf
Info in <TCanvas::Pri

In [5]:
# Clean efficiency, unclean rejection and clean purity of the kept events for a few cut values
# (none: purity = clean fraction of the sample). The purity is computed from efficiency and rejection rather than
# from the purity graph, which has no points (and would be extrapolated) where no events are kept.
def purity(e, r, n_c, n_u):
    kept = e * n_c + (1. - r) * n_u
    return f"{e * n_c / kept:.3f}" if kept > 0 else "n/a"

for var, cfg in VARIABLES.items():
    print(f"=== {var}: cut -> clean kept / unclean rejected / purity")
    for (v, sample, level), (eff, rej, n_c, n_u) in scans.items():
        if v != var:
            continue
        row = [f"none: purity {purity(1., 0., n_c, n_u)}"]
        for cut in cfg["cuts"]:
            e, r = eff.Eval(cut), rej.Eval(cut)
            row.append(f"{cut:+.1f}: {e:.3f}/{r:.3f}/{purity(e, r, n_c, n_u)}")
        print(f"{sample:28s} {level:6s} " + "  ".join(row))

=== y23: cut -> clean kept / unclean rejected / purity
wzp6_ee_nunuH_Hbb_ecm240     parton none: purity 0.955  -3.0: 0.286/0.997/1.000  -2.5: 0.472/0.979/0.998  -2.0: 0.673/0.889/0.992  -1.5: 0.854/0.666/0.982  -1.0: 0.971/0.271/0.966
wzp6_ee_nunuH_Hbb_ecm240     hadron none: purity 0.955  -3.0: 0.336/0.998/1.000  -2.5: 0.507/0.980/0.998  -2.0: 0.692/0.889/0.993  -1.5: 0.862/0.663/0.982  -1.0: 0.973/0.266/0.966
wzp6_ee_nunuH_Hcc_ecm240     parton none: purity 0.949  -3.0: 0.234/0.991/0.998  -2.5: 0.443/0.949/0.994  -2.0: 0.667/0.839/0.987  -1.5: 0.854/0.607/0.976  -1.0: 0.970/0.236/0.960
wzp6_ee_nunuH_Hcc_ecm240     hadron none: purity 0.949  -3.0: 0.290/0.992/0.998  -2.5: 0.489/0.945/0.994  -2.0: 0.690/0.832/0.987  -1.5: 0.863/0.603/0.976  -1.0: 0.972/0.234/0.959
wzp6_ee_nunuH_Hdd_ecm240     parton none: purity 0.946  -3.0: 0.223/0.990/0.998  -2.5: 0.436/0.935/0.992  -2.0: 0.665/0.824/0.985  -1.5: 0.854/0.596/0.974  -1.0: 0.971/0.223/0.957
wzp6_ee_nunuH_Hdd_ecm240     hadron none: pur

## Energy cut scan for the inclusive jet multiplicity

`<level>_incl_ecut_vs_njets_{clean,unclean}`: x = jet energy cut, y = number of inclusive anti-kt jets
(ee_genkt, R = 0.5) with E > cut, one entry per event and cut value.

* Top pad: clean (blue) and unclean (red) drawn as boxes on top of each other. Each class is normalised to its
  own number of events (every column sums to 1 per class) and both use the same maximum, so the boxes compare
  the shapes, not the 5 % vs. 95 % sizes.
* Bottom pad: for the selection N_jets = 2 at each energy cut, the fraction of clean events kept, of unclean
  events rejected, and the clean fraction (purity) of the kept events.
* Output: `plots/<level>_ifn/njets_ecut_scan/<sample>.pdf` and `all_samples_njets_ecut_scan.pdf`.

In [6]:
NJETS_SEL = 2  # selection evaluated in the bottom pad: N_jets == NJETS_SEL
NJETS_MAX = 12  # y range shown


def draw_ecut_scan(tfile, sample, level):
    name = f"{level}_incl_ecut_vs_njets"
    h_c = tfile.Get(f"{name}_clean").Clone(f"{sample}_{level}_ecut_c")
    h_u = tfile.Get(f"{name}_unclean").Clone(f"{sample}_{level}_ecut_u")
    nx = h_c.GetNbinsX()
    iy = h_c.GetYaxis().FindFixBin(NJETS_SEL)
    # events per class = entries in one column (every event fills each column once)
    n_c = h_c.Integral(1, 1, 0, h_c.GetNbinsY() + 1)
    n_u = h_u.Integral(1, 1, 0, h_u.GetNbinsY() + 1)

    eff, rej, pur = ROOT.TGraph(), ROOT.TGraph(), ROOT.TGraph()
    for ix in range(1, nx + 1):
        x = h_c.GetXaxis().GetBinLowEdge(ix)  # the cut value
        k_c = h_c.GetBinContent(ix, iy)
        k_u = h_u.GetBinContent(ix, iy)
        eff.SetPoint(eff.GetN(), x, k_c / n_c if n_c else 0.)
        rej.SetPoint(rej.GetN(), x, 1. - k_u / n_u if n_u else 0.)
        if k_c + k_u > 0:
            pur.SetPoint(pur.GetN(), x, k_c / (k_c + k_u))

    for h, n in ((h_c, n_c), (h_u, n_u)):
        h.SetDirectory(0)
        if n > 0:
            h.Scale(1. / n)
    hmax = max(h_c.GetMaximum(), h_u.GetMaximum())
    for h, col in ((h_c, ROOT.kBlue + 1), (h_u, ROOT.kRed + 1)):
        h.SetMinimum(0.)
        h.SetMaximum(hmax)
        h.SetLineColor(col)
        h.SetLineWidth(1)
        h.SetFillStyle(0)

    xmin, xmax = h_c.GetXaxis().GetXmin(), h_c.GetXaxis().GetXmax()
    c = ROOT.TCanvas(f"{level}_ecut_{sample}", "", 700, 800)
    top = ROOT.TPad("top", "", 0, 0.38, 1, 1)
    bot = ROOT.TPad("bot", "", 0, 0, 1, 0.38)
    top.SetBottomMargin(0.02)
    bot.SetTopMargin(0.03)
    bot.SetBottomMargin(0.25)
    for p in (top, bot):
        p.SetGridx()
        p.Draw()

    top.cd()
    h_c.SetTitle(f"{sample}, {level} level;;N_{{jets}} (ee_genkt R=0.5, E > cut)")
    h_c.GetXaxis().SetLabelSize(0)
    h_c.GetYaxis().SetRangeUser(0., NJETS_MAX + 1.)
    h_c.Draw("box")
    h_u.Draw("box same")
    leg = ROOT.TLegend(0.45, 0.72, 0.88, 0.88)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    leg.SetTextSize(0.045)
    leg.AddEntry(h_c, f"clean parton flavour ({n_c / (n_c + n_u):.1%})", "l")
    leg.AddEntry(h_u, f"not clean ({n_u / (n_c + n_u):.1%})", "l")
    leg.Draw()

    bot.cd()
    frame = bot.DrawFrame(xmin, 0, xmax, 1.05)
    frame.SetTitle(f";jet energy cut [GeV];fraction (N_{{jets}} = {NJETS_SEL})")
    for ax, s in ((frame.GetXaxis(), 0.08), (frame.GetYaxis(), 0.08)):
        ax.SetLabelSize(s)
        ax.SetTitleSize(s)
    frame.GetXaxis().SetTitleOffset(1.2)
    frame.GetYaxis().SetTitleOffset(0.55)
    frame.GetYaxis().SetNdivisions(505)
    eff.SetLineColor(ROOT.kBlue + 1)
    rej.SetLineColor(ROOT.kRed + 1)
    pur.SetLineColor(ROOT.kGreen + 2)
    pur.SetLineStyle(2)
    for g in (eff, rej, pur):
        g.SetLineWidth(2)
        g.Draw("L same")
    leg2 = ROOT.TLegend(0.55, 0.3, 0.88, 0.65)
    leg2.SetBorderSize(0)
    leg2.SetFillColorAlpha(ROOT.kWhite, 0.8)  # the curves cross the whole pad
    leg2.SetTextSize(0.08)
    leg2.AddEntry(eff, "clean kept", "l")
    leg2.AddEntry(rej, "not clean rejected", "l")
    leg2.AddEntry(pur, "clean fraction of kept", "l")
    leg2.Draw()

    outdir = os.path.join(PLOT_DIR, f"{level}_ifn", "njets_ecut_scan")
    os.makedirs(outdir, exist_ok=True)
    c.SaveAs(os.path.join(outdir, f"{sample}.pdf"))
    c.Draw()
    c._keep = (top, bot, h_c, h_u, eff, rej, pur, leg, leg2, frame)
    return c, eff, rej, pur, n_c, n_u

In [7]:
ecut_canvases = {}
ecut_scans = {}
for fname in files:
    sample = os.path.basename(fname)[:-5]
    tfile = ROOT.TFile.Open(fname)
    for level in LEVELS:
        c, eff, rej, pur, n_c, n_u = draw_ecut_scan(tfile, sample, level)
        ecut_canvases.setdefault(level, []).append(c)
        ecut_scans[(sample, level)] = (eff, rej, n_c, n_u)
    tfile.Close()

for level, cs in ecut_canvases.items():
    combined = os.path.join(PLOT_DIR, f"{level}_ifn", "njets_ecut_scan", "all_samples_njets_ecut_scan.pdf")
    cs[0].Print(combined + "[")
    for c in cs:
        c.Print(combined, f"Title:{c.GetName()}")
    cs[-1].Print(combined + "]")

# N_jets == NJETS_SEL for a few energy cuts: clean kept / unclean rejected / purity
print(f"=== N_jets == {NJETS_SEL} vs. energy cut: clean kept / unclean rejected / purity")
for (sample, level), (eff, rej, n_c, n_u) in ecut_scans.items():
    row = []
    for ecut in [5., 10., 15., 20., 25., 30.]:
        e, r = eff.Eval(ecut), rej.Eval(ecut)
        row.append(f"{ecut:.0f}: {e:.3f}/{r:.3f}/{purity(e, r, n_c, n_u)}")
    print(f"{sample:28s} {level:6s} " + "  ".join(row))

Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_ecut_scan/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_ecut_scan/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_ecut_scan/wzp6_ee_nunuH_Hcc_ecm240.pdf has been created


Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_ecut_scan/wzp6_ee_nunuH_Hcc_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_ecut_scan/wzp6_ee_nunuH_Hdd_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_ecut_scan/wzp6_ee_nunuH_Hdd_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_ecut_scan/wzp6_ee_nunuH_Hgg_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_ecut_scan/wzp6_ee_nunuH_Hgg_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_ecut_scan/wzp6_ee_nunuH_Hss_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_ecut_scan/wzp6_ee_nunuH_Hss_ecm240.pdf has been created
Info in <TCanvas::Pr

Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_ecut_scan/wzp6_ee_nunuH_Huu_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_ecut_scan/all_samples_njets_ecut_scan.pdf has been created
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_ecut_scan/all_samples_njets_ecut_scan.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_ecut_scan/all_samples_njets_ecut_scan.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_ecut_scan/all_samples_njets_ecut_scan.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_ecut_scan/all_samples_njets_ecut_scan.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/njets_ecut_scan

Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_ecut_scan/all_samples_njets_ecut_scan.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_ecut_scan/all_samples_njets_ecut_scan.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_ecut_scan/all_samples_njets_ecut_scan.pdf
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/hadron_ifn/njets_ecut_scan/all_samples_njets_ecut_scan.pdf has been closed


=== N_jets == 2 vs. energy cut: clean kept / unclean rejected / purity
wzp6_ee_nunuH_Hbb_ecm240     parton 5: 0.440/0.987/0.999  10: 0.620/0.913/0.993  15: 0.734/0.790/0.987  20: 0.824/0.610/0.978  25: 0.887/0.420/0.970  30: 0.907/0.330/0.967
wzp6_ee_nunuH_Hbb_ecm240     hadron 5: 0.533/0.983/0.999  10: 0.671/0.906/0.994  15: 0.762/0.784/0.987  20: 0.840/0.605/0.979  25: 0.898/0.423/0.971  30: 0.918/0.318/0.966
wzp6_ee_nunuH_Hcc_ecm240     parton 5: 0.427/0.957/0.995  10: 0.613/0.859/0.988  15: 0.732/0.717/0.980  20: 0.823/0.539/0.971  25: 0.885/0.379/0.964  30: 0.900/0.303/0.960
wzp6_ee_nunuH_Hcc_ecm240     hadron 5: 0.524/0.941/0.994  10: 0.667/0.840/0.987  15: 0.763/0.702/0.979  20: 0.840/0.536/0.971  25: 0.896/0.373/0.964  30: 0.914/0.291/0.960
wzp6_ee_nunuH_Hdd_ecm240     parton 5: 0.424/0.936/0.991  10: 0.609/0.839/0.985  15: 0.728/0.698/0.977  20: 0.822/0.521/0.968  25: 0.884/0.371/0.961  30: 0.902/0.301/0.958
wzp6_ee_nunuH_Hdd_ecm240     hadron 5: 0.520/0.917/0.991  10: 0.662/0